## Tasks

4.1 predict magnitude of antibody response - H1N1 A/Victoria/4897/2022 (D28)

4.2 predict magnitude of antibody response - H3N2 A/Massachusetts/18/2022 (D28)

4.3 predict magnitude of antibody response - Vic B/Austria/1359417/2021 (D28)

4.4 predict magnitude of antibody response - all 3 vaccine strains (D28)

**4.5 predict antibody breadth - all variants (D28)**
* Training Data: Demographics + Day 0 + Day 7 innate
* Assay: HAI
* Measure: Geo mean
* Metric: Spearman correlation
* Full description: Geomean HAI across all variants

4.6 predict antibody breadth - all variants (D28)

4.7 predict antibody durability - H1N1 A/Victoria/4897/2022 (D365)

4.8 predict antibody durability - H3N2 A/Massachusetts/18/2022 (D365)

4.9 predict antibody durability - Vic B/Austria/1359417/2021 (D365)

4.10 predict antibody durability - all 3 vaccine strains (D365)

### Task 4.5: Focus
Breadth refers to how widely an antibody response covers different variants of a pathogen (not just the specific strain, but also related versions).
Want to measure against various strains and summarize them together.
Use the geometric mean (average used for antibody titers)

In [1]:
import pandas as pd

In [2]:
DATA_PATH = 'data/PART2-26-01-26_reorg/PART2-26-01-26_reorg'
train_hai = pd.read_csv(DATA_PATH + '/train_hai.tsv', sep='\t')
train_participants = pd.read_csv(DATA_PATH + '/train_participants.tsv', sep='\t')

tables = {
    'train_hai': train_hai,
    'train_participants': train_participants,
}

In [3]:
for name, df in tables.items():
    print(f"\n{'=' * 50}")
    print(f'TABLE: {name}')
    print(f"{'=' * 50}")
    display(df.head(5))


TABLE: train_hai


,hai_id,participant_id,timepoint,virus_strain,value,material
0,ID_001__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_001,0.0,H1N1 A/South Carolina/1/1918,20.,Unknown
1,ID_001__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_001,28.0,H1N1 A/South Carolina/1/1918,40.,Unknown
2,ID_002__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_002,0.0,H1N1 A/South Carolina/1/1918,5.,Unknown
3,ID_002__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_002,28.0,H1N1 A/South Carolina/1/1918,5.,Unknown
4,ID_003__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_003,0.0,H1N1 A/South Carolina/1/1918,80.,Unknown



TABLE: train_participants


,participant_id,subject,biological_sex,race,min_age,max_age,geolocation,investigation_id,investigation_name,arm_id,arm_name,data_source,description,basic_curation,pubmed_ids,main_pmid,main_publication_author
0,SDY269.SUB112836,SUB112836,female,White,28,28,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
1,SDY269.SUB112849,SUB112849,female,Black or African American,39,39,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
2,SDY269.SUB112854,SUB112854,male,Black or African American,46,46,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
3,SDY269.SUB112860,SUB112860,female,White,32,32,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)
4,SDY269.SUB112881,SUB112881,female,Black or African American,29,29,US: Georgia,SDY269,Systems Biology of 2008 Influenza Vaccination ...,ARM1888,LAIV group 2008,"ImmPort, ImmuneSpace",Healthy adults given 2008 LAIV vaccine,no,21743478 26682988,21743478,Nakaya HI (2011)


#### Data Cleaning
HAI
* Timepoint column only missing 1.89% of values, and value column missing 0.02% of values. We decide to drop these rows.

Participants
* main_publication_author missing 47.55% of values, geolocation missing 20.18%, pubmed_ids missing 3.06%, main_pmid missing 3.06%

In [4]:
# For HAI we can dropna as the amount of missing values is low
train_hai.dropna()
# For Participants drop publication column all together
train_participants.drop(columns=['pubmed_ids'], inplace=True)

In [5]:
train_merged = train_hai.merge(train_participants, on='participant_id', how='left')

cols_to_keep = [
    'hai_id', 'participant_id', 'timepoint', 'virus_strain', 'value', 'biological_sex', 'race', 'min_age',
    'geolocation', 'investigation_id', 'investigation_name', 'arm_id', 'arm_name', 'data_source', 'description',
]

train_merged = train_merged[cols_to_keep]
train_merged.head()

,hai_id,participant_id,timepoint,virus_strain,value,biological_sex,race,min_age,geolocation,investigation_id,investigation_name,arm_id,arm_name,data_source,description
0,ID_001__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_001,0.0,H1N1 A/South Carolina/1/1918,20.,female,race: unknown,29,US: Georgia,2016_UGA,2016_UGA_Fluzone,2016_UGA_Standard_Fluzone,2016 UGA Standard Fluzone,UGA,2016 UGA Standard Fluzone
1,ID_001__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_001,28.0,H1N1 A/South Carolina/1/1918,40.,female,race: unknown,29,US: Georgia,2016_UGA,2016_UGA_Fluzone,2016_UGA_Standard_Fluzone,2016 UGA Standard Fluzone,UGA,2016 UGA Standard Fluzone
2,ID_002__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_002,0.0,H1N1 A/South Carolina/1/1918,5.,female,race: unknown,29,US: Georgia,2016_UGA,2016_UGA_Fluzone,2016_UGA_Standard_Fluzone,2016 UGA Standard Fluzone,UGA,2016 UGA Standard Fluzone
3,ID_002__2016_UGA_Standard_Fluzone__21__HAI__H1...,2016_UGA.ID_002,28.0,H1N1 A/South Carolina/1/1918,5.,female,race: unknown,29,US: Georgia,2016_UGA,2016_UGA_Fluzone,2016_UGA_Standard_Fluzone,2016 UGA Standard Fluzone,UGA,2016 UGA Standard Fluzone
4,ID_003__2016_UGA_Standard_Fluzone__0__HAI__H1N...,2016_UGA.ID_003,0.0,H1N1 A/South Carolina/1/1918,80.,female,race: unknown,28,US: Georgia,2016_UGA,2016_UGA_Fluzone,2016_UGA_Standard_Fluzone,2016 UGA Standard Fluzone,UGA,2016 UGA Standard Fluzone


In [6]:
train_merged.columns

Index(['hai_id', 'participant_id', 'timepoint', 'virus_strain', 'value',
       'biological_sex', 'race', 'min_age', 'geolocation', 'investigation_id',
       'investigation_name', 'arm_id', 'arm_name', 'data_source',
       'description'],
      dtype='object')